In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier


In [2]:
df = pd.read_csv("adult.csv")

In [3]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
df["workclass"].unique()

array(['?', 'Private', 'State-gov', 'Federal-gov', 'Self-emp-not-inc',
       'Self-emp-inc', 'Local-gov', 'Without-pay', 'Never-worked'],
      dtype=object)

In [5]:
df["education"].unique()

array(['HS-grad', 'Some-college', '7th-8th', '10th', 'Doctorate',
       'Prof-school', 'Bachelors', 'Masters', '11th', 'Assoc-acdm',
       'Assoc-voc', '1st-4th', '5th-6th', '12th', '9th', 'Preschool'],
      dtype=object)

In [6]:
df["marital.status"].unique()

array(['Widowed', 'Divorced', 'Separated', 'Never-married',
       'Married-civ-spouse', 'Married-spouse-absent', 'Married-AF-spouse'],
      dtype=object)

In [7]:
df.drop(["fnlwgt",
"education.num",
"relationship",
"race",
"capital.gain",
"capital.loss",
"native.country"],axis=1,inplace=True)

In [9]:
df.head()

,age,workclass,education,marital.status,occupation,sex,hours.per.week,income
0,90,?,HS-grad,Widowed,?,Female,40,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,Female,18,<=50K
2,66,?,Some-college,Widowed,?,Female,40,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,Female,40,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,Female,40,<=50K


In [8]:
df.isnull().sum()

age               0
workclass         0
education         0
marital.status    0
occupation        0
sex               0
hours.per.week    0
income            0
dtype: int64

In [26]:
# ? itself is missing value bro

df.replace("?", np.nan,inplace=True)

In [27]:
df.head()

,age,workclass,education,marital.status,occupation,sex,hours.per.week,income
0,90,NaN,HS-grad,Widowed,NaN,Female,40,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,Female,18,<=50K
2,66,NaN,Some-college,Widowed,NaN,Female,40,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,Female,40,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,Female,40,<=50K


In [10]:
df.isnull().sum()

age               0
workclass         0
education         0
marital.status    0
occupation        0
sex               0
hours.per.week    0
income            0
dtype: int64

In [11]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=["income"]),
                                                df["income"],
                                                test_size = 0.2,
                                                random_state = 0
                                                )

In [36]:
X_train.shape

(26048, 7)

In [13]:
# now first apply simple imputer

trf1 = ColumnTransformer([

    ("workclass_imputer",SimpleImputer(strategy="most_frequent"),[1]),
    ("occupation_imputer",SimpleImputer(strategy="most_frequent"),[4])

    
],remainder="passthrough") # this gives updated column index 0,1,3,4,5

In [12]:
# now encoding needed 

trf2 = ColumnTransformer([

    ("ohe",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[0,1,3,4,5])
    
],remainder="passthrough")

In [14]:
trf3 = DecisionTreeClassifier() # now 3rd model 

In [15]:
pipe = Pipeline([

    ("trf1",trf1),
    ("trf2",trf2),
    ("trf3",trf3),

])

In [16]:
pipe.fit(X_train,y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('workclass_imputer', ...), ('occupation_imputer', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
pipe.named_steps["trf1"].transformers_[0][1].statistics_

array(['Private'], dtype=object)

In [74]:
pipe.named_steps["trf1"].transformers_[1][1].statistics_

array(['Prof-specialty'], dtype=object)

In [75]:
y_pred = pipe.predict(X_test) # jo values model ne nhi dekhi means testing values us ki prediction

In [76]:
y_pred

array(['<=50K', '<=50K', '<=50K', ..., '>50K', '<=50K', '<=50K'],
      shape=(6513,), dtype=object)

In [77]:
from sklearn.metrics import accuracy_score
# y test jo ke actual answers tha X_test vali rows ke or y pred jo ke model ne pedict kr ke diya
accuracy_score(y_test,y_pred)

0.787655458314141

In [78]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe,X_train,y_train,cv=5,scoring="accuracy").mean()

np.float64(0.7830540674286973)

In [18]:
from sklearn.model_selection import GridSearchCV

params = {

    "trf3__criterion": ["gini", "entropy", "log_loss"],
    "trf3__max_depth": [2, 3, 5, 10, None],
    "trf3__min_samples_split": [2, 5, 10],
    "trf3__min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    
    estimator=pipe,
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf3__criterion': ['gini', 'entropy', ...], 'trf3__max_depth': [2, 3, ...], 'trf3__min_samples_leaf': [1, 2, ...], 'trf3__min_samples_split': [2, 5, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('workclass_imputer', ...), ('occupation_imputer', ...)]"


In [19]:
grid.best_params_

{'trf3__criterion': 'entropy',
 'trf3__max_depth': 10,
 'trf3__min_samples_leaf': 4,
 'trf3__min_samples_split': 10}

In [21]:
grid.best_score_

np.float64(0.826320597489433)

In [23]:
import pickle

pickle.dump(pipe,open("emp_pred.pkl","wb"))